### Globalne ustawienia

In [ ]:
import os

Struktura folderu
- labs2
    - case3_single.fastq.gz
    - copy_files
    - FastQC_reports
    - reports_case3
        - initial_report
        - trimming

In [ ]:
# ustawienie agenta ssh
agent = !echo $SSH_AUTH_SOCK
os.environ["SSH_AUTH_SOCK"] = agent[0] # to trzeba wydobyć
sv = 'ssh lamina'

# ustawienia fastp
js_set = '--disable_json'

# ustawienia ścieżek
fastq_file = 'labs2/case3_single.fastq.gz' # adres pliku
write_path = 'labs2/reports_case3' # adres folderu z raportami
copy_path = 'raporty' # adres lokalnego folderu dla plików 

### Raport
#### Opis pomiaru [\[1\]](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE96961), [\[2\]](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2546850)
- gatunek: mus musculus
- typ analizy: RNA-seq 
- instrument: Illumina HiSeq 2500
- układ biblioteki: pojedyncza
- wybór biblioteki: cDNA 
- źródło biblioteki: transkryptomiczna 
- długość odczytu: 65
- tkanka: Hipokamp 
#### Analiza wstępna
##### Analiza %GC
Według informacji z [\[3\]](https://bionumbers.hms.harvard.edu/bionumber.aspx?id=102409&ver=6), powinniśmy się spodziewać $\%\mathrm{GC}  \approx 47.36$, ponieważ tkanka pochodzi z hipokampu. W naszym pomiarze jest na poziomie 49%, może to wskazywać na kontaminacje. Jest to małe odchylenie, jednak niepokojący jest średni rozkład %GC po wszystkich sekwencjach, wskazuje to na nad reprezentacje pomiarową.

Zasoby z case3_single_fastqc.html:
- *Basic Statistics*
- *Per sequence GC content*

##### Stan klastrów
Widzimy, że kilka klastrów, to jest 2001-2005, 2001-2003 ma gorszą jakość odczytów. 

Zasoby z case3_single_fastqc.html:
- *Per tile sequence quality*

##### Jakość odczytu
Jakość odczytu jest bardzo dobra. Każdy cykl ma wysoki wskaźnik Phred'a oraz rozkład średniej jakości odczytu jest skoncentrowany na wartości 38-40 w skali Phred'a. Nie doszło do problemów ze zbyt krótkimi sekwencjami, zatem nie mamy do czynienia z pozostawionymi adapterami na końcach odczytów. Prawidłowa jest także długość każdego odczytu, rozkład histogramu jest trójkątny, wynika to z uciąglenia histogramu, którego bin przypada na każdy cykl

Zasoby z case3_single_fastqc.html:
- *Per base sequence quality*
- *Per sequence quality scores*
- *Per base N content*
- *Adapter Content*
- *Sequence Length Distribution*

##### Duplikacja i nadreprezentacja 
Mamy znacząco zaburzone rozkłady zasad na poszczególnych cyklach. Jednocześnie widzimy nad reprezentacje sekwencji
$$
\mathrm{GATCGGAAGAGCACACGTCTGAACTCCAGTCACAGTCAACAATCTCGTATGCCGTCTTCTGCTTG} \approx 48.2\%
$$
W dokumentacji protokołu [\[2\]](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2546850), możemy znaleźć informacje o tym, że został użyty adapter z prefiksem:
$$
\mathrm{GATCGGAAGAGCACACGTCTGAACTCCAGTCAC}
$$
W dokumentacji illuminy możemy znaleźć, że podana sekwencja jest adapterem [\[4\]](https://support-docs.illumina.com/SHARE/AdapterSequences/Content/SingleIndexes.htm). Mamy duży poziom duplikacji (30%), wynika to z typu analiza, która bada poziom ekspresji genów, nie jest niepokojący.

Zasoby z case3_single_fastqc.html:
- Per base sequence content
- Sequence Duplication Levels
#### Wnioski
Jakość odczytów jest bardzo dobra, podejrzewam zatem, że zaburzenie reprezentacji rozkładów, może wynikać z nadreprezentacji adapterów, czego przyczyną mogła być źle przygotowana biblioteka. 


### Bibliografia
[1](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE96961). “Series GSE96961.” Nih.gov, 2024, www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE96961. Accessed 19 Oct. 2025.4

[2](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2546850). “Sample GSM2546850.” Nih.gov, 2024, www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2546850. Accessed 19 Oct. 2025.Schulze-Hentrich, Julia . 

[3](https://bionumbers.hms.harvard.edu/bionumber.aspx?id=102409&ver=6). Moran, Uri. “GC Content - Mouse Mus Musculus - BNID 102409.” Harvard.edu, 2025, bionumbers.hms.harvard.edu/bionumber.aspx?id=102409&ver=6. Accessed 19 Oct. 2025.

[4](https://support-docs.illumina.com/SHARE/AdapterSequences/Content/SingleIndexes.htm). “TruSeq Single Indexes.” Illumina.com, 2025, support-docs.illumina.com/SHARE/AdapterSequences/Content/SingleIndexes.htm. Accessed 19 Oct. 2025.

In [ ]:
# wstępna analiza
loc_write_path = write_path + '/initial_report'
!{sv} "fastqc {fastq_file} -o {loc_write_path}"

Sprawdzamy czy są podane sekwencje UMI. Korzystamy ze znajomości formatu plików i będziemy przechodzić co 4 linie, żeby uzyskać informacje z samych nagłówków:

Struktura:
1. Nagłówek  - sekwencja UMI znajduje się na końcu po "UMI:
2. Sekwencja
3. Separator 
4. Jakości

In [ ]:
!{sv} "zcat  {fastq_file} | sed -n '1~4p' | grep -o "UMI:[A-Z]*""

Nie otrzymujemy żadnych wyników, zatem UMI nie były wykorzystane 

### Raport 
#### Trimming 
##### Metodologia 
Zakładamy, że sekwencje adapterów zaczynają się od:
$$
\mathrm{GATCGGAAGAGCACACGTCTGAACTCCAGTCAC}
$$
i używamy funkcji `fastp --adapter_sequence`.
##### Wyniki
zasoby z trimming_manual_fastqc.html
- *Basic Statistics*
- *Per base sequence content*
- *Per sequence GC content*
- *Sequence Duplication Levels*
- *Overrepresented sequences*
##### Analiza wyników
Widzimy, że teraz %GC jest bliżej wartości z literatury oraz, że jej średni rozkład po wszystkich sekwencjach jest zbliżony do rozkładu normalnego. Rozkłady zasad po cyklu 10 nie są zaburzone. Zaburzenie wcześniejsze wynika z typu analizy. Ponieważ w danych nie ma sekwencji UMI, nie mamy podstaw do ucięcia początku sekwencji. Mamy do czynienia z dużą duplikacją na poziomie 60%. Nie jestem jednak w stanie stwierdzić czy jest to wyniki niepokojący - możemy mieć do czynienia z komórką o niskiej różnorodności transkryptów. Żeby to zbadać, musiałbym porównać więcej komórek o tym profilu.

Nie usunąłem także kolejnych nad reprezentowanych sekwencji. Wynika to z tego, że nie jestem pewien czy są to rzeczywiście RNA PCR jak sugeruje fastp. 


**Komentarz** 
Sprawdziłem także, czy wyniki z trimmingu będą się różnić, jeśli
- wykonamy go automatycznie (bez podania polecenia)
- wprowadzimy ręcznie prefiks znaleziony w dokumentacji
W tym przypadku, oba są analogiczne, jednak będziemy korzystać z ręcznego trimmingu

W tym przypadku są takie same, zostawiam skrypt tylko na manualne - mamy pewność że nie usunie istotnych biologicznie sekwencji

In [ ]:
## ścieżki
loc_file_name = write_path + '/trimming_manual'
loc_write_path = write_path + '/trimming'
## informacyjnie jest to sekwencja która najczęściej występowała
# adapter_detected_sequence = "GATCGGAAGAGCACACGTCTGAACTCCAGTCACAGTCAACAATCTCGTATGCCGTCTTCTGCTTG"
## jest to prefix adapterów użytych w bibliotece 
adapter_universal_prefix = "GATCGGAAGAGCACACGTCTGAACTCCAGTCAC"

# usuwamy wszystkie adaptery z danym prefiksem 
!{sv} "fastp -i '{fastq_file}' \
    -o '{loc_file_name}.fastq.gz' --html '{loc_file_name}.html' --json '{loc_file_name}.json'\
    --adapter_sequence '{adapter_universal_prefix}'" 

# powtórzenie raportu
!{sv} "fastqc {loc_file_name}.fastq.gz -o {loc_write_path}" 


In [ ]:
# Ściągnięcie danych
!scp -r lamina:'{write_path}/*' {copy_path} 